Ask a mathematician whether an optimal agent exists for this competition and the answer arrives in one line, from a theorem published in 1928. The game is finite, both players move at once, and exactly one of them wins. Von Neumann's minimax theorem settles it, and tells you nothing you can use.

Ask instead whether a **winning** strategy exists, one that beats every opponent more often than it loses, and the answer is **no**, for a reason worth three minutes of your time. The game has random inputs, and section 2 shows they do not rescue it. And once you accept that, the useful questions turn out to be quantitative rather than existential:

**When does the game actually start?** Every episode of this competition begins in an identical position, and the engine's first random draw lands at a specific turn. Before it, nothing in your observation differs between one game and another, so **every optimal agent plays a fixed script on that prefix**, whether or not it has ever seen another player's code.

**Is this a competitive game at all?** Two banks in the same episode correlate at **+0.73**, and the correlation survives controlling for who is playing. Most of what determines your result also determines your opponent's.

**And what shape is the thing you are actually betting on?** Not the bank. The margin, which has an excess kurtosis of twelve and fails every normality test on the board, and yet predicts win rates to within seven points.

None of this is a strategy. It is the shape of the box a strategy has to fit inside. The concrete side, what the market can pay and which crop the field over-grows, is a separate question with its own notebook, referenced at the end.

Measured on 32,570 real episodes from Georgy Mamarin's [Kaggriculture Episodes](https://www.kaggle.com/datasets/georgymamarin/kaggriculture-episodes), and derived from the public [engine source](https://github.com/Kaggle/kaggle-environments/tree/master/kaggle_environments/envs/kaggriculture). Companion notebooks, each on one question: [Wins, not money](https://www.kaggle.com/code/destbreso/wins-not-money) on what to maximise, [Everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) on how alike the field's schedules are, [The leaderboard is a habitat gradient](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient) on the board as a population. Competition: [Kaggriculture](https://www.kaggle.com/competitions/kaggriculture).

In [1]:
# Chart style, matched to the companion notebooks: recessive axes, a light
# horizontal grid, nothing decorative. Colour is assigned by the job it does.
import csv, math, os
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e7e7e4'
plt.rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 130,
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10,
    'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'text.color': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white'})

def tidy(ax, ylab=None, title=None):
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.xaxis.grid(False)
    if ylab: ax.set_ylabel(ylab)
    if title: ax.set_title(title, loc='left', pad=10, fontweight='bold')
    return ax

def find_data(roots=('/kaggle/input',)):
    """Find the dataset by WALKING for episodes.csv, at whatever depth it sits.

    Kaggle does not mount an input at a fixed depth: it can appear as
    /kaggle/input/<slug>/ or as /kaggle/input/datasets/<owner>/<slug>/.
    A list of candidate paths is how the first version of this notebook failed,
    silently, with BASE = None and a TypeError three lines later."""
    def scan(root, max_depth=6):
        root = os.path.abspath(root)
        base = root.rstrip(os.sep).count(os.sep)
        hits = []
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count(os.sep) - base >= max_depth:
                dirnames[:] = []
                continue
            dirnames[:] = [d for d in dirnames if not d.startswith('.')]
            if 'episodes.csv' in filenames:
                hits.append(dirpath)
        return hits

    found = []
    for r in roots:
        if os.path.isdir(r):
            found += scan(r)
    here = os.path.abspath(os.getcwd())          # local checkout: walk upward
    while not found:
        for sub in ('data', '.'):
            d = os.path.join(here, sub)
            if os.path.isdir(d):
                found += scan(d, max_depth=4)
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    if found:
        return found[0]
    # Fail loudly and say what was actually there, rather than returning None
    # and letting the error surface somewhere unrelated.
    seen = []
    for r in roots:
        if os.path.isdir(r):
            seen += [os.path.join(dp, f) for dp, _, fs in os.walk(r) for f in fs][:40]
    raise FileNotFoundError(
        'episodes.csv not found. Attach the dataset '
        'georgymamarin/kaggriculture-episodes to this notebook.\n'
        'What is mounted:\n  ' + ('\n  '.join(seen) if seen else '(nothing)'))

BASE = find_data()

# The ablation in section 2 replays one fixed route against itself. Any route
# works; this one is a recorded stream published with the competition.
ROUTE_PATH = next((p for p in ('agents/v7.6_zeddouk/main.py',
                               '../agents/v7.6_zeddouk/main.py',
                               '../../agents/v7.6_zeddouk/main.py')
                   if os.path.exists(p)), None)
N_ABLATE = 40                     # 80 episodes; offline this was run at 300

# Two cells run the simulator rather than reading the dataset. The import is
# guarded because a notebook that dies on it shows the reader nothing at all,
# and nothing else here depends on it.
try:
    from kaggle_environments import make
    ENGINE = True
except ImportError:
    ENGINE = False
    print('kaggle_environments is unavailable, so the two engine demonstrations '
          'are skipped.\nAttach the Kaggriculture competition to run them.')

# Every number below is computed from this, live. Nothing is pasted in.
rows = []
with open(BASE + '/episodes.csv') as f:
    for r in csv.DictReader(f):
        if r['type'] != 'EPISODE_TYPE_PUBLIC' or r['state'] != 'COMPLETED':
            continue                     # validation episodes are self-play
        try:
            x, y = float(r['bank_0']), float(r['bank_1'])
        except (TypeError, ValueError):
            continue
        rows.append((r['create_time'] or '', r['sub_0'], r['sub_1'], x, y))
# Chronological, because section 5 measures a SERIAL correlation and the order
# of the file is not guaranteed to be the order the games were played in.
rows.sort(key=lambda t: t[0])            # ISO 8601, so lexicographic works
sub0 = [r[1] for r in rows]; sub1 = [r[2] for r in rows]
b0 = np.array([r[3] for r in rows]); b1 = np.array([r[4] for r in rows])
M = np.r_[b0, b1]                        # one agent's bank, pooled over seats
Dm = b0 - b1                             # the signed margin
print(f'{BASE}\n{len(b0):,} public completed episodes, {len(set(sub0)|set(sub1)):,} submissions')

## 1. What kind of game this is

Worth pinning down before quoting any theorem at it, because the theorems that apply are decided entirely by these four properties.

**Finite.** 720 turns, a finite action set, a finite state.

**Simultaneous.** Both agents submit turn $t$ without seeing the other's move, and contested market units are quoted against the same pre-commit inventory for both.

**Stochastic, but very thinly.** The engine draws random numbers in exactly two places, both at the end of a day, both from a generator keyed on the seed and the day: weeds spawning on empty tiles, and a town shop unlocking every three days. **The starting state uses neither**, so every episode of this competition begins in an identical position.

Those two draws share one generator, and section 3.1 shows that this is not the harmless implementation detail it looks like.

**And almost perfectly observed**, which is the property most likely to be misremembered. Each agent's observation contains **both farms in full**: the opponent's tiles, money, farmer position, hands and unlocked land. Hidden is exactly one thing, the opponent's private block, meaning their shed contents and seed stock. Since tiles are visible and the shed is fed by harvests from visible tiles, even that is largely inferable.

So this is **not a partially observed game in any serious sense**, which matters: the hard complexity results about decentralised partially observed control do not bite here.

One more property, and it is the one that makes the rest of the notebook necessary. **The game is zero-sum in its outcome and general-sum in its money.** The leaderboard scores the sign of your bank minus theirs, but both of you can and usually do make money. It is a zero-sum preference laid over a cooperative-looking production economy.

That shape has a name and a literature. Immorlica and co-authors call it a **dueling game** and bound how much optimising the absolute objective costs you on the relative one [3]. It is the reason a notebook on what to maximise had to exist separately from this one.

## 2. An optimal agent exists. A winning strategy does not.

**The first half is free.** For a finite two-player zero-sum game, von Neumann's minimax theorem gives a value and a pair of optimal strategies [1]; Kuhn extends it to sequential play, showing that with perfect recall behavioural strategies are as powerful as mixed ones [2]. Kaggriculture satisfies both hypotheses, and nothing about farming enters the argument. It is a corollary, not a result, and its only use is to close the question.

**The second half has content, and it needs care, because this game has random inputs.**

The classical statement is that in a **symmetric** zero-sum game the value is zero. The argument is one line: if a strategy did better than a coin flip against every opponent, then by symmetry it would do better than a coin flip against a copy of itself, and two copies of the same thing cannot both be above half.

Randomness does not break that argument, but it does change what the conclusion is allowed to say. Both halves are worth stating exactly.

**What the randomness does not do.** Symmetry here is a claim about distributions, not about any single episode, and the engine supplies it. The two farms are **byte-identical at turn 0**, same money, same unlocked quadrant, same starting square, in every episode ever played, because the opening state is built from constants and never touches the seed. The town is shared, so both players see the same shops. Weeds are genuinely random and are drawn for seat 0 before seat 1 from one generator, so the seats receive **different numbers**, but from the same distribution. The cell below checks both.

**What the randomness does do.** It deletes the word *guarantee*. No strategy can guarantee a win in a single episode, whatever it is, because an unlucky enough draw exists. So the claim is about the average: **no strategy can hold a win probability above one half against every opponent.**

**And one asymmetry is not random at all.** The market loop commits seat 0's unit before seat 1's, which looks like a first-mover advantage and is the obvious place for one to hide. It is not one, and that can be settled by reading the source rather than measured.

Every failure condition in the per-unit commit reads the acting player's **own** state and nothing else: a sale fails if that player's shed is empty, a purchase fails if that player's money is short or that player's shed is full. The only shared object the commit mutates is the market's inventory, **and no failure condition reads it.** Both players were quoted for this unit against the same pre-commit inventory before either committed, so the price is already fixed for both. Committing first therefore changes neither player's price and cannot make the other's order fail. On the next pass through the loop both are re-quoted against the same new inventory.

The same holds for the weeds. Seat 0's are drawn first, but the count each seat consumes depends on its own bare tiles, so the **total** consumed before the shop is chosen is the same whichever seat goes first, and the shop that opens is identical under a swap.

So the symmetry is derived, not assumed. What follows is the verification, and it is worth running anyway: a derivation about an engine is a claim about a specific version of it.

In [2]:
# Does the randomness treat the two seats alike? Two checks, because two
# different things could break the symmetry the argument needs.
if ENGINE:
    # (a) the opening. If the farms differed at turn 0, symmetry would be dead
    #     before the first action.
    env = make('kaggriculture', configuration={'episodeSteps': 30, 'seed': 7})
    env.reset(2)
    f = env.state[0].observation['farms']
    import json as _json
    same = (_json.dumps(f[0], sort_keys=True, default=str)
            == _json.dumps(f[1], sort_keys=True, default=str))
    print(f'the two farms are byte-identical at turn 0: {same}')
    print(f"  money {f[0]['money']:.0f} / {f[1]['money']:.0f}   "
          f"farmer {f[0]['farmer']} / {f[1]['farmer']}   "
          f"land {f[0]['unlocked_quadrants']} / {f[1]['unlocked_quadrants']}")

    # (b) the weeds. Drawn seat 0 first, then seat 1, from ONE generator, so the
    #     two seats get different numbers. Symmetry needs the same DISTRIBUTION,
    #     not the same numbers, so that is what gets tested.
    def season_weeds(seed):
        e = make('kaggriculture', configuration={'episodeSteps': 720, 'seed': int(seed)})
        e.run(['pass', 'pass'])
        o = e.state[0].observation
        return [sum(1 for row in o['farms'][p]['tiles'] for t in row
                    if isinstance(t, dict) and t.get('kind') == 'WEED') for p in (0, 1)]

    W = np.array([season_weeds(s) for s in
                  np.random.default_rng(3).integers(1, 10 ** 6, 60)])
    print(f'\nweeds at the end of the season, over {len(W)} seeds:')
    print(f'  seat 0   mean {W[:, 0].mean():5.2f}   sd {W[:, 0].std():4.2f}')
    print(f'  seat 1   mean {W[:, 1].mean():5.2f}   sd {W[:, 1].std():4.2f}')
    print(f'  paired t-test on the difference   p = '
          f'{stats.ttest_rel(W[:, 0], W[:, 1]).pvalue:.3f}')
    print(f'  two-sample KS on the distributions p = '
          f'{stats.ks_2samp(W[:, 0], W[:, 1]).pvalue:.3f}')
    print('\n-> different draws, same distribution, which is what the argument needs.')

    # (c) and how different? The engine spawns weeds for seat 0 and then seat 1
    #     from ONE generator inside the same loop, so the two boards consume
    #     disjoint stretches of the same stream. Same distribution, different
    #     squares. This is why two identical routes do not always tie.
    def layout(seed):
        e = make('kaggriculture', configuration={'episodeSteps': 720, 'seed': int(seed)})
        e.run(['pass', 'pass'])
        fm = e.state[0].observation['farms']
        return [{(x, y) for y, row in enumerate(fm[p]['tiles'])
                 for x, t in enumerate(row)
                 if isinstance(t, dict) and t.get('kind') == 'WEED'} for p in (0, 1)]

    print('\nthe SAME agent in both seats, where the weeds actually landed:')
    ndiff = 0
    for i, sd in enumerate((11, 22, 33, 44, 55, 66, 77, 88)):
        a, b = layout(sd)
        ndiff += a != b
        if i < 2:
            print(f'  seed {sd}: seat 0 {sorted(a)}')
            print(f'           seat 1 {sorted(b)}')
    print(f'  layouts differ in {ndiff} of 8 seeds')

    # (d) the one structural asymmetry, tested rather than argued: seat 0's unit
    #     commits before seat 1's. If that were worth anything, identical orders
    #     would cost different amounts.
    def buyer(n):
        def a(obs, cfg=None):
            if obs['day'] == 0 and obs['hour'] == 1:
                return {'farmer': ['PASS'], 'hands': [],
                        'market': [['BUY_PRODUCT', 'WHEAT', n]]}
            return {'farmer': ['PASS'], 'hands': [], 'market': []}
        return a

    def spend(n0, n1):
        e = make('kaggriculture', configuration={'episodeSteps': 6, 'seed': 99})
        e.run([buyer(n0), buyer(n1)])
        fm = e.state[0].observation['farms']
        return 3000 - fm[0]['money'], 3000 - fm[1]['money']

    print('\nboth seats buying the same good on the same turn:')
    for n in (1, 5, 20, 60):
        a, b = spend(n, n)
        print(f'  {n:>2} units each   seat 0 spends {a:>6,.0f}   seat 1 spends {b:>6,.0f}'
              f'   {"identical" if a == b else "DIFFERENT"}')
    for n0, n1 in ((10, 2), (2, 10)):
        a, b = spend(n0, n1)
        print(f'  {n0:>2} against {n1:<2}    seat 0 {a/n0:>5.1f}/unit        '
              f'seat 1 {b/n1:>5.1f}/unit')
    print('  -> the per-unit price follows the ORDER SIZE, and swapping the sizes')
    print('     swaps the cost exactly. Committing first is worth nothing.')

**The opening is not random, the randomness is exchangeable, and committing first is worth nothing.** The derivation above said the last of those; the test says it too, and the two disagreeing would have been the interesting outcome.

**But notice what the weed layouts show, because it matters two sections later.** The two boards do not merely draw from the same distribution, they draw *different numbers*: the engine spawns for seat 0 and then seat 1 from one generator in the same loop, so each board consumes a different stretch of the same stream. Same agent, same episode, different squares under weed, and sometimes a different count of them. **Two identical routes are therefore not playing identical games**, which is exactly why they do not always tie.

There is a test of the premise sharper than anything on the ladder, and the dataset already contains it. **Every submission, before it is allowed onto the leaderboard, plays one verification game against itself.** It is unrated, it does not touch anyone's score, and it happens exactly once for every submission that has ever entered.

That makes it the argument's own hypothetical, run for real and run as a census. The strategy is held fixed and the only thing that varies is which seat it sits in. And because there is one episode per submission rather than many for a few, it probes symmetry **across the strategy space** rather than at one point in it.

In [3]:
# stream_hashes.csv fingerprints each seat's action stream at five prefixes, so
# two seats playing the same route in the same episode collide by construction.
import collections
HS = collections.defaultdict(dict)
with open(BASE + '/stream_hashes.csv') as f:
    for r in csv.DictReader(f):
        HS[r['episode_id']][r['seat']] = r
pub = set()
with open(BASE + '/episodes.csv') as f:
    for r in csv.DictReader(f):
        if r['type'] == 'EPISODE_TYPE_PUBLIC' and r['state'] == 'COMPLETED':
            pub.add(r['episode_id'])

COLS = [('stream_h24', 'the first day'), ('stream_h100', 'turn 100'),
        ('stream_h200', 'turn 200'), ('stream_h400', 'turn 400'),
        ('stream_h719', 'the whole game')]
n, same = 0, {c: 0 for c, _ in COLS}
for eid in pub:
    d = HS.get(eid)
    if not d or '0' not in d or '1' not in d:
        continue                       # replay coverage trails the crawl rate
    n += 1
    for c, _ in COLS:
        if d['0'][c] and d['0'][c] == d['1'][c]:
            same[c] += 1

print(f'{n:,} of {len(pub):,} public episodes have BOTH seats fingerprinted')
print(f'  ({100*n/len(pub):.0f} % coverage, set by the dataset crawler rather than by any')
print('   choice here: replays.parquet holds fewer episodes than the hashes do)\n')
print(f"{'the two seats play identically through':<40}{'episodes':>10}{'share':>9}")
for c, lab in COLS:
    print(f'{lab:<40}{same[c]:>10,}{100*same[c]/n:>8.1f} %')

# And what actually HAPPENS in a full mirror? Not a coin flip.
mir, rest = [], []
with open(BASE + '/episodes.csv') as f:
    for r in csv.DictReader(f):
        if r['type'] != 'EPISODE_TYPE_PUBLIC' or r['state'] != 'COMPLETED':
            continue
        d = HS.get(r['episode_id'])
        if not d or '0' not in d or '1' not in d:
            continue
        try:
            a, b = float(r['bank_0']), float(r['bank_1'])
        except (TypeError, ValueError):
            continue
        h = d['0']['stream_h719']
        (mir if h and h == d['1']['stream_h719'] else rest).append(a - b)
mir, rest = np.array(mir), np.array(rest)
print(f'\n{"":<26}{"mirrors":>10}{"every other episode":>22}')
print(f'{"episodes":<26}{len(mir):>10,}{len(rest):>22,}')
print(f'{"tie to the dollar":<26}{100*(mir==0).mean():>9.0f} %{100*(rest==0).mean():>21.1f} %')
print(f'{"median absolute margin":<26}{np.median(np.abs(mir)):>10,.0f}'
      f'{np.median(np.abs(rest)):>22,.0f}')

fig, ax = plt.subplots(figsize=(8.0, 3.6))
labs = [lab for _, lab in COLS]
vals = [100 * same[c] / n for c, _ in COLS]
ax.bar(range(len(vals)), vals, color=[ORANGE] + [BLUE] * 4, width=.62)
for i, v in enumerate(vals):
    ax.text(i, v + .5, f'{v:.1f} %', ha='center', fontsize=9,
            fontweight='bold', color=ORANGE if i == 0 else INK)
ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, fontsize=9)
ax.set_ylim(0, max(vals) * 1.25)
ax.set_xlabel('identical up to')
tidy(ax, 'share of public episodes', 'How often you are playing your own route')
plt.tight_layout(); plt.show()

**One public episode in eighty-three is a byte-identical mirror for all 719 turns**, and one in five shares an entire opening day. Not near-copies: the same stream, hashed.

That is not surprising once you look at how this field is built. Much of it plays fixed recorded routes, strong routes get published and resubmitted, so a route that works spreads and a route that has spread is one you meet.

**And the outcome of a mirror is sharper than the theorem promises.** The argument above says a strategy cannot beat a copy of itself more than half the time. What actually happens is sharper than that and worse for you: **they tie to the dollar in 99 % of them.** So few separate at all that the split among those few carries no information and is not quoted. Everywhere else on the board an exact tie happens in well under one per cent of games, with a median margin in the thousands.

**Identical actions are not quite enough for an identical result**, and it is worth being exact about why rather than assuming. In the self-play sample below, episodes where both seats played the same stream tie 90 % of the time rather than 100 %.

The obvious suspect is the weeds, and the engine makes that testable rather than arguable. **There are three sources of weeds and only one is random**: a plant past its lifespan decays into one, a plant left unwatered two days becomes one, and both of those are deterministic in the agent's own actions. Only the spawn rolls, once per bare tile, for seat 0 and then seat 1 from a single generator.

`weedSpawnChance` is a configuration key, so it can be switched off. If the spawn is the only asymmetric channel, identical routes must then tie **every** time.

In [4]:

# THE ROUTE COMES FROM THE DATASET, not from a file next to the notebook. An
# earlier version read one out of the author's repository, which exists in
# exactly one place on earth, so on Kaggle it resolved to None and the cell
# raised. replays.parquet is already attached, so lift a real 719-turn action
# stream out of it and replay that in both seats.
import json as _json

N_ABLATE = 40          # 80 episodes; offline this was run at 300

def load_route():
    import pyarrow.parquet as pq
    best = None
    with open(BASE + '/episodes.csv') as f:
        for r in csv.DictReader(f):
            if r['type'] != 'EPISODE_TYPE_PUBLIC' or r['state'] != 'COMPLETED':
                continue
            try:
                b = float(r['bank_0'])
            except (TypeError, ValueError):
                continue
            if best is None or b > best[1]:
                best = (r['episode_id'], b)
    t = pq.read_table(BASE + '/replays.parquet',
                      filters=[('episode_id', '==', int(best[0]))],
                      columns=['replay_json'])
    steps = _json.loads(t.column('replay_json')[0].as_py())['steps']
    return [None] + [(s[0].get('action') if s else None) for s in steps], best

PASS = {'farmer': ['PASS'], 'hands': [], 'market': []}

def mirror(route, seed, weed):
    e = make('kaggriculture', configuration={'episodeSteps': 720,
                                             'seed': int(seed),
                                             'weedSpawnChance': weed})
    e.reset(2)
    st = 0
    while not e.done:
        a = route[st + 1] if st + 1 < len(route) else None
        act = a if isinstance(a, dict) else dict(PASS)
        e.step([dict(act), dict(act)])        # the SAME stream in both seats
        st += 1
    o = e.state[0].observation
    w = [sum(1 for row in o['farms'][p]['tiles'] for t in row
             if isinstance(t, dict) and t.get('kind') == 'WEED') for p in (0, 1)]
    return float(e.state[0].reward or 0) - float(e.state[1].reward or 0), w

if ENGINE and os.path.exists(BASE + '/replays.parquet'):
    ROUTE, src = load_route()
    print(f"replaying the stream from episode {src[0]}, which banked "
          f"${src[1]:,.0f}, against itself\n")
    sds = [int(x) for x in np.random.default_rng(20260815).integers(1, 10**6, N_ABLATE)]
    for lab, wc in (('the random spawn at its default, 0.005', 0.005),
                    ('the random spawn switched off, 0.0', 0.0)):
        out = [mirror(ROUTE, s, wc) for s in sds]
        v = np.array([o[0] for o in out])
        w = np.array([o[1] for o in out])
        print(f'  {lab:<40} ties {int((v==0).sum()):>3}/{len(v)}'
              f'  ({100*(v==0).mean():5.1f} %)   weeds '
              f'{w[:,0].mean():.2f}/{w[:,1].mean():.2f}')
    print('\n  with the spawn off the weeds that remain are the deterministic ones,')
    print('  and both farms get exactly the same ones.')
else:
    print('the engine or replays.parquet is unavailable, so the ablation is skipped')

**With the spawn off, every episode ties to the dollar.** Run at larger scale offline, 150 seeds on each of two different routes, it is 300 of 300.

So the random weed is the only thing that separates two identical routes, and the claim that started this section is now a measurement rather than a guess about a mechanism. It also fixes the size of what the environment alone is worth: a few hundred dollars, against a median decided margin on the board in the thousands.

That is the premise verified in the one configuration that matters, across hundreds of **different** strategies rather than repetitions of one. It is a smaller sample than the ladder and its interval is correspondingly wide, so the two are complementary: **self-play is the right experiment, the ladder is the powerful one.**

### The same game answers a question it was not asked

About half of these are decided rather than tied, far more than the mirrors above, and the reason is worth having.

**A fixed script cannot diverge from itself.** Whatever the weeds do, it plays the same actions in both seats. So if an agent's two seats finish an episode playing *different* streams, it reacted to something, and one free verification game per submission is enough to see it.

**So a mirror is not a coin flip, it is a dead heat, and a dead heat is a game you cannot gain from.** The more of the field plays your route, the larger the share of your own schedule that is guaranteed to return nothing. **An edge decays not only because the field improves, but because your edge becomes the field**, and the decay shows up as draws rather than as losses. That is a hard cap no amount of tuning removes.

There is one more caveat, and it is about the premise rather than the conclusion. The engine's symmetry is derived above from its source, so it is a claim about a specific version of that source. The checks below verify the derivation rather than substituting for it.

In [5]:
# Validation episodes are self-play. Every one is a strategy against a copy of
# itself, which is exactly the configuration the symmetry argument invokes.
v0, v1, ident, subs_seen = [], [], 0, set()
with open(BASE + '/episodes.csv') as f:
    for r in csv.DictReader(f):
        if r['type'] != 'EPISODE_TYPE_VALIDATION' or r['state'] != 'COMPLETED':
            continue
        try:
            x, y = float(r['bank_0']), float(r['bank_1'])
        except (TypeError, ValueError):
            continue
        ident += r['sub_0'] == r['sub_1']
        subs_seen.add(r['sub_0'])
        v0.append(x); v1.append(y)
v0, v1 = np.array(v0), np.array(v1)
dec = v0 != v1
w0 = int((v0 > v1).sum())
bt = stats.binomtest(w0, int(dec.sum()))
print(f'{len(v0)} validation episodes, {ident} of them the SAME submission on both seats')
print(f'  distinct submissions {len(subs_seen)}, so one verification game each')
print(f'  exact ties        {int((~dec).sum())}  ({100*(~dec).mean():.0f} %)')
print(f'  decided           {int(dec.sum())}')
print(f'  seat 0 wins       {w0} of {int(dec.sum())} = {100*w0/dec.sum():.1f} %')
print(f'  binomial p        {bt.pvalue:.3f}   '
      f'95 % interval [{100*bt.proportion_ci().low:.1f} %, '
      f'{100*bt.proportion_ci().high:.1f} %]')
print(f'  mean margin       {(v0-v1).mean():+,.0f} on a largest of '
      f'{np.abs(v0-v1).max():,.0f}')

**Nearly half of them tie to the dollar**, which is what two identical agents in an identical opening produce until a weed lands on one farm and not the other. Of the ones that are decided, seat 0 takes about half, and the test does not come close to rejecting.

That is the premise verified in the one configuration that matters, across hundreds of **different** strategies rather than repetitions of one. It is a smaller sample than the ladder and its interval is correspondingly wide, so the two are complementary: **self-play is the right experiment, the ladder is the powerful one.**

### The same game answers a question it was not asked

About half of these are decided rather than tied, far more than the mirrors above, and the reason is worth having.

**A fixed script cannot diverge from itself.** Whatever the weeds do, it plays the same actions in both seats. So if an agent's two seats finish an episode playing *different* streams, it reacted to something, and one free verification game per submission is enough to see it.

In [6]:
# The verification game doubles as a purity test: a fixed script plays the same
# actions in both seats whatever the weeds do, so a submission whose two seats
# diverge has reacted to something.
val = []
with open(BASE + '/episodes.csv') as f:
    for r in csv.DictReader(f):
        if r['type'] != 'EPISODE_TYPE_VALIDATION' or r['state'] != 'COMPLETED':
            continue
        d = HS.get(r['episode_id'])
        if not d or '0' not in d or '1' not in d:
            continue
        try:
            val.append((d, float(r['bank_0']) - float(r['bank_1'])))
        except (TypeError, ValueError):
            pass

print(f'{len(val)} verification games with both streams fingerprinted\n')
print(f"{'seats still identical at':<26}{'submissions':>13}{'share':>9}")
for c, lab in COLS:
    k = sum(1 for d, _ in val if d['0'][c] and d['0'][c] == d['1'][c])
    print(f'{lab:<26}{k:>13,}{100*k/len(val):>8.1f} %')

def _split(keep):
    return np.array([m for d, m in val
                     if keep(bool(d['0']['stream_h719'])
                             and d['0']['stream_h719'] == d['1']['stream_h719'])])
same_s, diff_s = _split(lambda x: x), _split(lambda x: not x)
print(f'\n{"":<34}{"episodes":>10}{"tie":>8}{"median |margin|":>18}')
for lab, v in (('seats played the same stream', same_s),
               ('seats diverged', diff_s)):
    print(f'{lab:<34}{len(v):>10,}{100*(v==0).mean():>7.0f} %'
          f'{np.median(np.abs(v)):>18,.0f}')
print(f'\n-> {100*len(diff_s)/len(val):.0f} % of submissions react to something. '
      f'The rest are effectively fixed.')

**Nearly every submission is still identical to itself at turn 24, and only about two in five still are at the end.** The rest diverged from themselves, which a script cannot do.

That is a measurement of how reactive this field is, taken from a game everyone is required to play, and it is cleaner than the usual approach of comparing two ladder episodes: those differ in opponent **and** in seed, so a difference between them proves nothing by itself. Here the agent is the same, the episode is the same, and the only thing that differs is the seat.

It also closes the gap between the two numbers above. Mirrors on the ladder tie almost always because both sides are fixed routes. Verification games tie only about half the time because most agents are not fixed, so their two seats genuinely play different games.

In [7]:
wins0, decided = int((b0 > b1).sum()), int((b0 != b1).sum())
bt = stats.binomtest(wins0, decided)
lo, hi = bt.proportion_ci()
print(f"seat 0 wins {wins0:,} of {decided:,} decided episodes = {100*wins0/decided:.2f} %")
print(f"  95 % interval  [{100*lo:.2f} %, {100*hi:.2f} %]   binomial p = {bt.pvalue:.3f}")
print(f"  ties: {len(b0)-decided:,} episodes")
print(f"\nmean margin {Dm.mean():+,.0f} on a spread of {Dm.std(ddof=1):,.0f}"
      f"   (t-test p = {stats.ttest_1samp(Dm, 0).pvalue:.3f},"
      f" Wilcoxon p = {stats.wilcoxon(Dm).pvalue:.3f})")

fig, ax = plt.subplots(figsize=(7.8, 2.4))
ax.plot([lo, hi], [0, 0], color=BLUE, lw=7, solid_capstyle='round', alpha=.55)
ax.plot([wins0/decided], [0], 'o', color=BLUE, ms=11, zorder=3)
ax.axvline(.5, color=ORANGE, lw=2)
ax.text(.5, .40, 'a coin flip', color=ORANGE, ha='center', fontsize=10, fontweight='bold')
ax.text(wins0/decided, -.42, f'{100*wins0/decided:.2f} %', color=BLUE, ha='center',
        fontsize=9, fontweight='bold')
ax.set_xlim(.485, .515); ax.set_ylim(-.7, .8); ax.set_yticks([])
ax.set_xlabel("seat 0's win rate, every decided episode on the board")
ax.spines['left'].set_visible(False)
tidy(ax, None, 'Going first is worth nothing')
ax.xaxis.grid(True, color=GRID, lw=.8); ax.yaxis.grid(False)
plt.tight_layout(); plt.show()

**The seat advantage is indistinguishable from zero**, with the interval comfortably containing a half over more than thirty thousand games.

So the game is symmetric, its value is zero, and **no winning strategy exists**: nothing can hold a win probability above one half against every opponent. What exists is an *unexploitable* strategy, worth roughly half your games against anybody, including whoever is currently first. Not a guarantee in any one episode, which the randomness rules out for everyone, but a floor on the average.

### Why that argument has teeth here, which is not obvious

**The opponent that defeats any strategy is a copy of itself.** That is the whole proof: nothing beats everyone, because *everyone* includes you.

The natural objection is that on a real leaderboard you never meet yourself, so the theorem forbids nothing about dominating the field that is actually there. **On this leaderboard that objection is wrong, and the dataset says by how much.**

### And unexploitable is the wrong thing to aim at

This is the practical consequence, and it is easy to get backwards.

A maximin strategy is safe against everything, which means it gives up the surplus available from a field that is predictable. A leaderboard pays for exactly that surplus. **Every strong agent on this board is a best response to the field as it currently is**, not a minimax solution, and that is the right choice for the objective.

It carries a specific liability, and it is the one to plan for: a best response decays as the field it responds to moves, and an unexploitable strategy would not. Whatever edge you have over the current field is a wasting asset with a half-life you should try to measure.

## 3. When the game starts

Here is a small theorem with a measurable constant.

Fix your action stream. Run it twice, changing only the episode seed, or only the opponent. If your observation is byte-identical up to turn $T$, then **no strategy could have acted differently up to turn $T$**, because conditioning on something that does not vary is conditioning on a constant. Formally: for any adaptive strategy there is an open-loop one agreeing with it on every history in that prefix, so on turns $0 \ldots T$ the value of adaptive play and of fixed play are **equal, exactly, not approximately**.

The first turn at which anything differs is therefore a lower bound on the first turn at which feedback can be worth a cent. Two engine constants set it, and both are computable without simulating anything.

In [8]:
# Shops unlock at the end of day d when (d+1) % 3 == 0, so the first is visible
# at the start of day 3. Two different seeds draw the same shop with p = 1/8.
TURNS_PER_DAY, N_SHOPS = 24, 8
FIRST_SHOP_TURN = 3 * TURNS_PER_DAY

# Weeds: every empty unlocked tile gets an independent 0.005 chance each night.
# Only the NW quadrant starts unlocked, so at most 25 tiles are eligible, fewer
# for a route that plants them. The horizon is a property of your route too.
WEED_P, NW = 0.005, 25
days = np.arange(1, 13)

fig, ax = plt.subplots(figsize=(8.4, 4.0))
for empty, col, lab in ((NW, BLUE, 'leaves all 25 starting tiles empty'),
                        (12, AQUA, 'plants half of them'),
                        (4, ORANGE, 'plants all but four')):
    ax.plot(days * TURNS_PER_DAY, 1 - (1 - WEED_P) ** (empty * days),
            color=col, lw=2.2, marker='o', ms=4, label=f'a route that {lab}')
ax.axvline(FIRST_SHOP_TURN, color=INK, lw=1.6, ls='--')
ax.text(FIRST_SHOP_TURN + 8, .90,
        f'turn {FIRST_SHOP_TURN}: the first shop unlocks,\n'
        f'and two seeds draw a different one\n'
        f'{100*(1-1/N_SHOPS):.0f} % of the time',
        fontsize=9, color=INK, fontweight='bold', va='top')
ax.set_xlabel('turn'); ax.set_ylim(0, 1); ax.set_xlim(0, 300)
ax.legend(frameon=False, fontsize=9, loc='lower right', bbox_to_anchor=(1, .02))
tidy(ax, 'chance a weed has appeared by now',
     'The only two channels through which a seed can become visible')
plt.tight_layout(); plt.show()

print(f"the shop channel opens at turn {FIRST_SHOP_TURN}, reliably")
print(f"the weed channel opens earlier but only for a route that leaves ground bare")

Running that as an experiment rather than a derivation, one fixed route replayed across six seeds and five distinct opponents over the full 720 turns, and recording the first turn at which each part of the observation differs:

| what first becomes visible | earliest turn | median | the channel |
|---|---|---|---|
| **anything at all** | **48** | 72 | a weed, on the opponent's land |
| **your own farm** | **72** | 168 | your own weeds, then the shop draw |
| **anything about your opponent** | **136** | 175 | market inventory moving under their sales |

Three consequences.

**The opening is a script for everyone, and not because anyone copied.** For the first 48 turns there is nothing in the observation that differs between one episode and another. Any agent that branches there is branching on a constant. That matters for reading [Everyone is playing the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening), which measures 94 % of the field sharing a schedule at turn 24: **turn 24 sits inside the horizon**, so part of that agreement is forced by the engine rather than chosen by the players. Copying may be happening too, but turn-24 agreement is not the evidence for it.

**Your opponent is invisible for the first fifth of the game.** You cannot tell who you are playing until turn 136 at the earliest, median 175, because the only channel that reveals them is the market inventory and it does not move until they sell something. By then a quarter of the episode is already committed. **Any plan that depends on identifying the opponent has to survive that delay.**

**And the horizon depends on your own route.** A route that plants every tile leaves no bare ground for weeds and so sees its first difference later than one that plants half. The horizon is a property of the pair, not of the game alone.

### 3.1 The shop draw is not exogenous, and this was a surprise

Everything above treats the shop unlock as the seed talking. It is not, and the reason is one line of the engine.

`_spawn_weeds` calls the generator **once per empty tile**, for both farms, and the shop is chosen from **the same generator immediately afterwards**. So the number of draws consumed before the shop is picked depends on how many tiles the two players have left bare. Plant more, consume fewer draws, and a different shop opens.

That is testable in four episodes.

In [9]:
# Two agents that differ ONLY in how much ground they leave bare. If the shop
# draw were a function of the seed, both would unlock the same shops.
def passer(obs, cfg=None):
    return {'farmer': ['PASS'], 'hands': [], 'market': []}

def planter(obs, cfg=None):
    priv = obs.get('private', {}) or {}
    f = obs['farms'][obs['player']]
    if (priv.get('seeds', {}) or {}).get('WHEAT', 0) > 0 \
            and f['tiles'][f['farmer'][1]][f['farmer'][0]] is None:
        return {'farmer': ['PLANT', 'WHEAT'], 'hands': [], 'market': []}
    if f['money'] >= 10:
        return {'farmer': ['EAST'], 'hands': [], 'market': [['BUY_SEED', 'WHEAT', 1]]}
    return {'farmer': ['PASS'], 'hands': [], 'market': []}

def episode(seed, a, b):
    env = make('kaggriculture', configuration={'episodeSteps': 720, 'seed': int(seed)})
    env.run([a, b])
    o = env.state[0].observation
    return list(o['town']['unlocked_shops']), 10000 - o['market']['inventory']['CARROT']

if ENGINE:
    print(f"{'seed':>6}{'shops identical?':>19}{'carrot demand, bare':>22}"
          f"{'carrot demand, planted':>25}")
    for sd in (11, 22, 33, 44):
        s_bare, d_bare = episode(sd, passer, passer)
        s_plant, d_plant = episode(sd, planter, planter)
        print(f"{sd:>6}{str(s_bare == s_plant):>19}{d_bare:>22,}{d_plant:>25,}")

**The shops differ in every seed, and the demand they create differs enormously.** Same episode seed, same engine, and the town wants a different set of goods depending only on how much bare ground the two farms left behind.

Three things follow, in increasing order of how much they matter.

**The seed does not determine your episode's economy.** It determines it jointly with both players' planting. You cannot look up a seed and know what the town will buy.

**It does not bias anything measured across many seeds.** Consuming a different number of draws before a uniform choice still gives a uniform choice: over forty seeds each way, the eight shops come up within a couple of points of one in eight under both play patterns, and the median demand per good barely moves. So the distribution is safe. What is not safe is any claim about a *particular* episode.

**And it is a strategic channel, however strange.** Leaving ground bare changes which shops open, for both players. Whether anyone can steer that deliberately is not established here and would need the mapping from draw count to outcome, which is a much harder problem than noticing the coupling. But the coupling is real, it is engine-exact, and it is a shared variable that **both** players push on, which makes it a better candidate than the seed alone for the correlation measured in the next section.

## 4. Is this competitive, or is it weather?

Two players share one market, so the intuition is that what one gains the other loses. The data says close to the opposite, and it changes how every score on this board should be read.

In [10]:
r_pool = stats.pearsonr(b0, b1)

# THE CONFOUND. Matchmaking pairs on rating, so two strong agents meet and both
# bank well for reasons having nothing to do with the market they share.
# Holding the PAIR fixed is not possible here: there are 30,135 distinct
# submission pairs in 32,570 episodes and the ladder almost never repeats a
# matchup. So we subtract each submission's own mean bank and correlate the
# residuals, which removes both agents' average strength using every episode.
tot, cnt = {}, {}
for s, x in list(zip(sub0, b0)) + list(zip(sub1, b1)):
    tot[s] = tot.get(s, 0.0) + x; cnt[s] = cnt.get(s, 0) + 1
ok = {s for s, n in cnt.items() if n >= 5}
keep = np.array([a in ok and b in ok for a, b in zip(sub0, sub1)])
e0 = np.array([x - tot[s]/cnt[s] for s, x in zip(sub0, b0)])
e1 = np.array([x - tot[s]/cnt[s] for s, x in zip(sub1, b1)])
r_within = stats.pearsonr(e0[keep], e1[keep])

pairs = set()
for a, b in zip(sub0, sub1):
    pairs.add((a, b) if a <= b else (b, a))

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.7))
i = np.random.default_rng(1).choice(len(b0), 6000, replace=False)
ax.scatter(b0[i], b1[i], s=5, color=BLUE, alpha=.18, lw=0)
lim = [0, np.percentile(np.r_[b0, b1], 99.5)]
ax.plot(lim, lim, color=MUTED, lw=1, ls=':')
m, c = np.polyfit(b0, b1, 1)
ax.plot(np.array(lim), m*np.array(lim)+c, color=ORANGE, lw=2)
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
ax.set_xlabel("seat 0's bank")
tidy(ax, "seat 1's bank", f'Both banks move together   r = {r_pool[0]:+.3f}')

j = np.random.default_rng(2).choice(int(keep.sum()), 6000, replace=False)
ax2.scatter(e0[keep][j], e1[keep][j], s=5, color=AQUA, alpha=.18, lw=0)
L = np.percentile(np.abs(np.r_[e0[keep], e1[keep]]), 99)
ax2.plot([-L, L], [-L, L], color=MUTED, lw=1, ls=':')
mm, cc = np.polyfit(e0[keep], e1[keep], 1)
ax2.plot([-L, L], [mm*-L+cc, mm*L+cc], color=ORANGE, lw=2)
ax2.set_xlim(-L, L); ax2.set_ylim(-L, L); ax2.set_aspect('equal')
ax2.set_xlabel("seat 0's bank, minus its own average")
tidy(ax2, "seat 1's bank, minus its own average",
     f'And still do with skill removed   r = {r_within[0]:+.3f}')
plt.tight_layout(); plt.show()

print(f"{len(pairs):,} distinct submission pairs in {len(b0):,} episodes: "
      f"the ladder almost never repeats a matchup\n")
print(f"pooled            r = {r_pool[0]:+.4f}")
print(f"within-submission r = {r_within[0]:+.4f}  "
      f"({int(keep.sum()):,} episodes where both sides have 5+ games)")
print(f"-> only {100*(r_pool[0]-r_within[0])/r_pool[0]:.0f} % of it was who happened to be playing")

The correlation survives the control almost untouched. **Most of what determines your bank also determines your opponent's**, and it is not that you were both good.

The likely mechanism is the one section 3.1 uncovered. Which shops unlock decides what the town will buy, both players farm into that same town, and **both of them influence the draw**. An episode that opens three pet cafes is a carrot episode for both of you at once. That is a hypothesis rather than a measurement: it is consistent with everything here and it is not what these numbers test.

There is a cross-check that costs nothing and would catch a spurious correlation on the spot. For two quantities of equal spread $\sigma$ and correlation $\rho$,

$$\mathrm{sd}(\text{margin}) = \sigma\sqrt{2(1-\rho)}$$

with nothing fitted. If the correlation is real, it has to reproduce the observed spread of the margin.

In [11]:
sig, rho = M.std(ddof=1), r_pool[0]
pred, obs = sig*math.sqrt(2*(1-rho)), Dm.std(ddof=1)
indep = sig*math.sqrt(2)
print(f"  sd of one bank                    {sig:>10,.0f}")
print(f"  predicted sd of the margin        {pred:>10,.0f}")
print(f"  observed  sd of the margin        {obs:>10,.0f}   "
      f"({100*abs(pred-obs)/obs:.1f} % apart)")
print(f"  if the two banks were independent {indep:>10,.0f}")
print(f"\n  the shared episode deletes {100*(1-obs/indep):.0f} % of the spread")

fig, ax = plt.subplots(figsize=(8.4, 3.9))
n, edges, _ = ax.hist(Dm, bins=140, color=BLUE, alpha=.85, edgecolor='white', lw=.3,
                      label=f'the margin as it actually is, sd {obs:,.0f}')
w = edges[1] - edges[0]
xs = np.linspace(Dm.min(), Dm.max(), 700)
ax.plot(xs, len(Dm)*w*stats.norm.pdf(xs, 0, indep), color=ORANGE, lw=2.2,
        label=f'if the two banks were independent, sd {indep:,.0f}')
ax.set_xlabel("seat 0's bank minus seat 1's")
ax.legend(frameon=False, fontsize=9)
tidy(ax, 'episodes', "The episode's own difficulty cancels out of the difference")
plt.tight_layout(); plt.show()

Predicted and observed agree to a rounding error, so the correlation is not an artefact of how the sample was assembled.

**And this is a mechanical argument for scoring on wins rather than banks.** Roughly half the variance in a bank is the episode's own difficulty, shared with your opponent, and it cancels in the difference. A margin is a sharper instrument than a bank precisely because the common half is deleted. Comparing banks across episodes keeps all of that noise; comparing them within one throws it away.

If you are tuning an agent by watching its average bank, you are watching a quantity that is about half weather.

## 5. The shape of the thing you are betting on

Win probability is a statement about the margin, so the margin's distribution is the object that matters, and it is not the one people usually plot.

A histogram of the **winner's** bank looks convincingly bell-shaped. That is not evidence of anything, because a maximum is skewed by construction: take two independent normals and the larger of the pair is not normal. Whatever bell shape the winner's histogram shows is the parent's, seen through a transform that has to distort it.

In [12]:
W, Lo = np.maximum(b0, b1), np.minimum(b0, b1)
g = np.random.default_rng(7)
ctrl = np.maximum(g.normal(M.mean(), M.std(), len(b0)),
                  g.normal(M.mean(), M.std(), len(b0)))

fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.8))
for ax, (v, lab, col) in zip(axes, [(W, "the winner's bank", BLUE),
                                    (M, "one agent's bank", AQUA),
                                    (Dm, 'the margin', ORANGE)]):
    ax.hist(v, bins=80, color=col, alpha=.85, edgecolor='white', lw=.3)
    ax.set_xlabel(lab)
    tidy(ax, 'episodes' if ax is axes[0] else None,
         f'skew {stats.skew(v):+.2f}   excess kurtosis {stats.kurtosis(v):+.2f}')
plt.tight_layout(); plt.show()

print("CONTROL, the max of two INDEPENDENT NORMALS with this mean and spread:")
print(f"  skew {stats.skew(ctrl):+.3f}, and it fails D'Agostino at "
      f"p = {stats.normaltest(ctrl)[1]:.1e}")
print("  so a bell-shaped winner's histogram says nothing about the parent.\n")
print(f"{'':<21}{'skew':>9}{'excess kurtosis':>18}")
for v, lab in ((M, "one agent's bank"), (W, "the winner's bank"),
               (Lo, "the loser's bank"), (Dm, 'the margin')):
    print(f"{lab:<21}{stats.skew(v):>9.3f}{stats.kurtosis(v):>18.3f}")
print(f"\nstandard error of skew at n = {len(Dm):,} is {math.sqrt(6/len(Dm)):.4f},")
print("so a skew of 0.1 is ten standard errors and still invisible on a chart.")

**The margin has an excess kurtosis around +12** against 0 for a normal, while the banks themselves are near symmetric and slightly light-tailed. It is by far the heaviest-tailed quantity here.

At this sample size hypothesis tests stop carrying information. **Every normality test rejects every one of these**, which is a fact about n = 32,570 and not about the data. The Q-Q plot is the honest picture, because it shows *where* the deviation lives.

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.9))
for ax, (v, lab) in zip(axes, [(W, "the winner's bank"), (M, "one agent's bank"),
                               (Dm, 'the margin')]):
    z = (v - v.mean()) / v.std(ddof=1)
    q = np.linspace(.0005, .9995, 900)
    ax.plot(stats.norm.ppf(q), np.quantile(z, q), color=BLUE, lw=2)
    ax.plot([-4, 4], [-4, 4], color=ORANGE, lw=1.4, ls='--')
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
    ax.set_xlabel('normal quantile')
    tidy(ax, 'observed quantile' if ax is axes[0] else None, lab)
plt.tight_layout(); plt.show()

import warnings
sm = np.random.default_rng(3)
print(f"{'':<21}{'D Agostino p':>15}{'Anderson A2':>14}{'Shapiro p':>13}")
with warnings.catch_warnings():
    warnings.simplefilter('ignore')      # scipy deprecation on anderson's API
    for v, lab in ((M, "one agent's bank"), (W, "the winner's bank"),
                   (Dm, 'the margin')):
        print(f"{lab:<21}{stats.normaltest(v)[1]:>15.2e}"
              f"{stats.anderson(v, 'norm').statistic:>14,.1f}"
              f"{stats.shapiro(sm.choice(v, 4500, replace=False)).pvalue:>13.2e}")
print("\nAll of them reject. That is the sample size talking, not the data.\n")

# WHERE does the heavy tail come from? The obvious guess is agents that broke.
# Test it rather than assert it: compare the loser's bank in the extreme
# margins against the loser's bank everywhere.
Lo_all = np.minimum(b0, b1)
big = np.abs(Dm) > 3 * Dm.std(ddof=1)
for thr in (1_000, 10_000):
    print(f"  loser banks under ${thr:,}:  {100*(Lo_all[big] < thr).mean():5.1f} % of the "
          f"{int(big.sum()):,} margins beyond 3 sd,  against "
          f"{100*(Lo_all < thr).mean():.1f} % of all episodes")

The banks track the diagonal closely and peel away only in the last half percent of each tail. The margin is straight through the middle and leaves the line hard at **both** ends, which is what an excess kurtosis of twelve looks like: **it is far too easy to win or lose by an enormous amount.**

The obvious guess is that the tail is agents that broke. The numbers above test it rather than assume it, and **the guess is mostly wrong.** Collapsed opponents are over-represented in the extreme margins by roughly threefold, which is real, but they account for only about a tenth of them. **Nine out of ten enormous margins are between two agents that both finished with a working farm.**

So the heavy tail is not an artefact to be cleaned away. Blowouts are a normal outcome of this game, and any model that treats the margin as thin-tailed will underestimate how often one is coming.

So the normal assumption fails. The question worth asking is what the failure **costs**, and there is only one thing the assumption is ever used for: turning a mean and a spread into a win probability. So test it exactly there, on every submission with enough games to have both.

In [14]:
# Built episode by episode, NOT seat 0's games then seat 1's, so that each
# submission's series stays in the order it was played. Section 5's last chart
# measures a serial correlation and would be wrong under any other order.
per_sub = {}
for a, b, d in zip(sub0, sub1, Dm):
    per_sub.setdefault(a, []).append(d)
    per_sub.setdefault(b, []).append(-d)

MIN_EP = 40
pred, obsr, shap = [], [], []
for s, v in per_sub.items():
    a = np.array(v, dtype=float)
    if len(a) < MIN_EP or a.std(ddof=1) == 0:
        continue
    pred.append(stats.norm.cdf(a.mean() / a.std(ddof=1)))
    obsr.append((a > 0).mean())
    shap.append(stats.shapiro(a).pvalue)
pred, obsr, shap = np.array(pred), np.array(obsr), np.array(shap)
err = obsr - pred

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.3))
ax.scatter(pred, obsr, s=24, color=BLUE, alpha=.55, lw=0)
ax.plot([0, 1], [0, 1], color=ORANGE, lw=1.6, ls='--')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
ax.set_xlabel('win rate the normal predicts')
tidy(ax, 'win rate actually achieved',
     f'{len(pred)} submissions with {MIN_EP}+ games   r = {stats.pearsonr(pred,obsr)[0]:.3f}')

ax2.hist(err, bins=34, color=AQUA, alpha=.88, edgecolor='white', lw=.4)
ax2.axvline(0, color=ORANGE, lw=1.8)
ax2.set_xlabel('observed minus predicted win rate')
tidy(ax2, 'submissions', f'Median absolute error {np.median(np.abs(err)):.3f}')
plt.tight_layout(); plt.show()

print(f"Shapiro-Wilk rejects normality for {100*(shap<0.05).mean():.1f} % of these "
      f"submissions (5 % is what an exactly normal world gives)")
print("and yet the normal approximation delivers:")
print(f"  correlation predicted vs achieved   {stats.pearsonr(pred, obsr)[0]:.3f}")
print(f"  mean signed error                   {err.mean():+.4f}")
print(f"  median absolute error               {np.median(np.abs(err)):.4f}")
print(f"  within 10 points                    {100*(np.abs(err)<.10).mean():.1f} %")

**The assumption fails the distributional test for almost every submission and still predicts the quantity it is used for to within about seven points.**

That is the sentence to quote, and it is far more useful than any p-value on this sample. It also says where not to trust it: the errors are largest exactly where the tail is heaviest, so a submission whose record includes several opponents that crashed will have its win rate systematically underestimated by the normal.

One last check, because every confidence interval anyone computes on a leaderboard win rate assumes episodes are independent draws.

In [15]:
lag1, runs_p = [], []
for s, v in per_sub.items():
    a = np.array(v, dtype=float)
    if len(a) < MIN_EP or a.std() == 0:
        continue
    lag1.append(np.corrcoef(a[:-1], a[1:])[0, 1])
    sg = a > 0
    n1, n0 = int(sg.sum()), int((~sg).sum())
    if n1 > 1 and n0 > 1:
        runs = 1 + int((sg[1:] != sg[:-1]).sum())
        mu = 2*n1*n0/(n1+n0) + 1
        var = (mu-1)*(mu-2)/(n1+n0-1)
        if var > 0:
            runs_p.append(2*stats.norm.sf(abs((runs-mu)/math.sqrt(var))))
lag1, runs_p = np.array(lag1), np.array(runs_p)

fig, ax = plt.subplots(figsize=(8.4, 3.8))
ax.hist(lag1, bins=34, color=BLUE, alpha=.88, edgecolor='white', lw=.4)
ax.axvline(0, color=MUTED, lw=1.2, ls=':')
ax.axvline(np.median(lag1), color=ORANGE, lw=2.2)
ax.text(np.median(lag1)+.02, ax.get_ylim()[1]*.86, f'median {np.median(lag1):+.2f}',
        color=ORANGE, fontweight='bold', fontsize=10)
ax.text(0.01, ax.get_ylim()[1]*.28, 'independent\nwould sit here', color=MUTED,
        fontsize=9, ha='left')
ax.set_xlabel("correlation between a submission's margin and the next one")
tidy(ax, 'submissions', 'Consecutive episodes are not independent draws')
plt.tight_layout(); plt.show()

print(f"median lag-1 autocorrelation of the margin: {np.median(lag1):+.3f}")
print(f"a runs test on the sign rejects independence for {100*(runs_p<0.05).mean():.1f} % "
      f"of submissions, against the 5 % expected if they were independent")

**Episodes of one submission are strongly serially correlated**, so a binomial or Wilson interval on a leaderboard win rate is too narrow. The likely mechanism is matchmaking: as a rating settles, consecutive games are drawn from a similar band of opponents, so they are not independent samples of the field.

The correction is the standard design effect for clustered samples, and the honest short version is that **your interval is wider than it looks and your effective number of games is smaller than your count.** If you are deciding between two submissions on a fifty-game difference, that is worth knowing.

## 6. What the box looks like

**A best agent exists and a winning strategy does not.** The opening is identical in every episode, the random inputs are drawn from the same distribution for both seats, and going first is worth nothing over thirty thousand games. So the value is a coin flip and nothing can stay above one half against everybody. What is available is an edge over the field as it currently stands, which is a wasting asset rather than a solution.

**The first fifty turns are identical in every game ever played, and your opponent is invisible for a fifth of the episode.** Adaptivity is worth exactly zero before turn 48 and nothing about the opponent before turn 136. Any design that leans on reading the opponent has to survive that delay first.

**The seed does not decide your economy.** Weeds and the shop unlock share one generator, and weeds are drawn once per bare tile, so **how much ground the two farms leave empty changes which shops open**. It does not bias anything averaged over seeds, and it means no particular episode's demand can be predicted from its seed.

**Half your result is the episode rather than you.** Two banks in one game correlate at +0.73 net of who is playing, and that shared half cancels out of the margin. This is why the margin is the sharper measurement and the average bank is a noisier one than it looks.

**And the margin is not normal, but the normal is good enough for the one job it has.** It fails every test and still lands win probabilities within about seven points, which is the number worth quoting instead of a p-value.

### What this does not answer

Whether the field's payoff structure has a stable attractor or cycles between counters. What identifying your opponent would actually be worth, given that you cannot do it before turn 136. And the concrete side entirely: how much money the market can pay at all, and which crop the field systematically over-grows, which is a separate question with its own arithmetic.

---

## References

Four, because only four are load-bearing. Each is the original source of a result this notebook leans on, rather than a survey that mentions it.

**[1]** von Neumann, J. (1928). *Zur Theorie der Gesellschaftsspiele.* **Mathematische Annalen 100**, 295-320. [EUDML](https://eudml.org/doc/159291). English translation in Tucker, A. W. & Luce, R. D. (eds.), *Contributions to the Theory of Games IV*, Princeton University Press (1959), 13-42.

> The minimax theorem: a finite two-player zero-sum game has a value, and both players have optimal strategies. This is what makes "does an optimal agent exist" a closed question in section 2 rather than an open one.

**[2]** Kuhn, H. W. (1953). *Extensive Games and the Problem of Information.* In Kuhn, H. W. & Tucker, A. W. (eds.), **Contributions to the Theory of Games, Volume II**, Annals of Mathematics Studies 28, Princeton University Press, 193-216. [DOI](https://doi.org/10.1515/9781400881970-012)

> Extends minimax to games played out over time, and proves that under **perfect recall** a behavioural strategy, one that picks an action given what you have seen, is as strong as any randomisation over complete plans. That is what licenses reasoning about turn-by-turn policies instead of whole 720-turn scripts, and it is also what makes section 3 airtight: a behavioural strategy facing an observation that does not vary has nothing to condition on, so it can do no better than a script.

**[3]** Immorlica, N., Kalai, A. T., Lucier, B., Moitra, A., Postlewaite, A. & Tennenholtz, M. (2011). *Dueling Algorithms.* **Proceedings of the 43rd ACM Symposium on Theory of Computing (STOC 2011)**, 215-224. [PDF](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/11/duel-stoc.pdf)

> Formalises the shape described in section 1: an algorithm optimised for absolute performance against one optimised to beat an opponent, with bounds on the gap. The reason "maximise your bank" and "maximise your win rate" are two different problems rather than one problem stated twice.

**[4]** Dubins, L. E. & Savage, L. J. (1965). *How to Gamble If You Must: Inequalities for Stochastic Processes.* McGraw-Hill. [Internet Archive](https://archive.org/details/inequalitiesfors0000dubi)

> Listed although not cited above, because it is the ancestor of a result the companion notebook develops. When a game is unfavourable and you must reach a target, **bold play maximises the probability of getting there**. Translated into this competition: when you are behind, dispersion helps you and a better median does not, which is the opposite of what an average-bank objective would recommend.

---

*Built on [Kaggriculture Episodes](https://www.kaggle.com/datasets/georgymamarin/kaggriculture-episodes) by Georgy Mamarin, whose daily refresh is what makes the empirical half of this possible, and on the public [engine source](https://github.com/Kaggle/kaggle-environments/tree/master/kaggle_environments/envs/kaggriculture). Competition: [Kaggriculture](https://www.kaggle.com/competitions/kaggriculture).*